# 三难度在线评测修复报告：可重复分析伴随 Notebook

本 Notebook 复用 `generate_artifact.py` 的同一组身份、哈希与数据投影检查，不维护第二套指标实现。

## tl;dr

`FINAL_LABEL` 默认保持 `None`，因此已提交执行结果只展示注册基线、历史诊断与语义 judge 审计，不把任何 candidate 自动认定为最终。负责人明确 final label 后，修改一个参数并从头执行即可得到同一 canonical report 数据。

## Context & Methods

Notebook 模式：technical report companion。读取 baseline registry、可选 final label run-index/artifacts、baseline failure diagnostics 与 semantic judge summary；所有输入都在仓库内，且由生成器验证 lineage 和 SHA-256。

### Key Assumptions

- Final label 必须由负责人显式批准，不能按目录名或更新时间推断。
- `passed` 是 archived evaluator 结果，不等于最终答案语义正确。
- 当前 diagnostics/judge summary 绑定原始 baseline failures，不能套用到未来 final candidate。
- 没有价格快照时 cost 保持 unknown。

In [1]:
from pathlib import Path
import importlib.util
import os

import pandas as pd

FINAL_LABEL = os.environ.get("STT_FINAL_REPORT_LABEL") or None  # 仅在负责人明确批准后设置

repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
report_dir = repo_root / "docs/evaluation-baselines/2026-08-10-three-difficulty/remediation-2026-08-10/technical-report"
generator_path = report_dir / "generate_artifact.py"
print(f"repo_root={repo_root}")
print(f"final_label={FINAL_LABEL!r}")

repo_root=D:\Learning\agent\STT
final_label=None


## Data

加载共享生成器并执行全部输入校验。若 final label 不完整、未配对或哈希不一致，本单元会失败而不会继续展示结果。

In [2]:
module_spec = importlib.util.spec_from_file_location("remediation_report_generator", generator_path)
report_generator = importlib.util.module_from_spec(module_spec)
assert module_spec.loader is not None
module_spec.loader.exec_module(report_generator)

report_data = report_generator.collect_report_data(FINAL_LABEL)
artifact = report_generator.build_artifact(report_data)
print({
    "surface": artifact["surface"],
    "snapshot_status": artifact["snapshot"]["status"],
    "final_label": FINAL_LABEL,
    "combined_input_sha256": report_data["inventory_hash"],
})

{'surface': 'report', 'snapshot_status': 'partial', 'final_label': None, 'combined_input_sha256': 'c3dc34b2a1e715a6ce474b2cc355f06efcecd3cf9cd1480d95e191107f672e7e'}


In [3]:
inventory = pd.DataFrame(report_data["datasets"]["input_inventory"])
display(inventory[["role", "difficulty", "path", "sha256"]])

,role,difficulty,path,sha256
0,artifact_copy_manifest,easy,…/​easy/​erun_​67606fd246654edb9a6b57ecfc4aab7...,d04fb8e0fe06ad96f4e1…
1,artifact_copy_manifest,hard,…/​hard/​erun_​54375b39653d4a2fb52b931f4dfbb01...,78a63f26847e7d56b093…
2,artifact_copy_manifest,medium,…/​medium/​erun_​eb7884462cd34655b1e703974f2d8...,0c4d9da7a065018acaac…
3,baseline_failure_diagnostics,all,…/​remediation-​2026-​08-​10/​diagnostics/​fai...,436086870470954f01e8…
4,baseline_registry,all,…/​2026-​08-​10-​three-​difficulty/​baselines/...,0b292fe4a68507d9e720…
5,diagnostics_api_results,easy,…/​erun_​67606fd246654edb9a6b57ecfc4aab79/​api...,aa0cce4c95bb4b14a073…
6,diagnostics_api_results,hard,…/​erun_​54375b39653d4a2fb52b931f4dfbb01c/​api...,ec9be7d3f51080cbaded…
7,diagnostics_api_results,medium,…/​erun_​eb7884462cd34655b1e703974f2d8662/​api...,0ada849ea5cd5f86fc78…
8,diagnostics_dataset_manifest,easy,docs/​evaluation-​baselines/​2026-​08-​10-​thr...,1a3cacbb59b3aa7f354a…
9,diagnostics_dataset_manifest,hard,docs/​evaluation-​baselines/​2026-​08-​10-​thr...,e5be4d2495618e819b97…


## Results

报告中的 stacked bar 使用 `case_outcomes`；Notebook 保留精确表格，避免重复生成第二张图或三行装饰性可视化。

In [4]:
outcomes = pd.DataFrame(report_data["datasets"]["case_outcomes"])
display(outcomes[["difficulty", "role", "passed", "failed", "error", "total"]])

,difficulty,role,passed,failed,error,total
0,简单,原始基线,28.0,22.0,0.0,50.0
1,中等,原始基线,21.0,29.0,0.0,50.0
2,困难,原始基线,9.0,41.0,0.0,50.0


In [5]:
quality = pd.DataFrame(report_data["datasets"]["quality_metrics"])
metric_columns = [
    "difficulty", "role", "pass_rate", "recall_at_1", "recall_at_3",
    "recall_at_10", "mrr", "ndcg_at_10", "correctness", "faithfulness",
    "citation_coverage", "citation_precision", "citation_recall",
    "abstention_accuracy", "abstention_fpr",
]
display(quality[metric_columns].style.format({column: "{:.1%}" for column in metric_columns[2:]}, na_rep="unknown"))

,difficulty,role,pass_rate,recall_at_1,recall_at_3,recall_at_10,mrr,ndcg_at_10,correctness,faithfulness,citation_coverage,citation_precision,citation_recall,abstention_accuracy,abstention_fpr
0,简单,原始基线,56.0%,80.0%,92.0%,94.0%,85.8%,87.9%,61.7%,85.7%,100.0%,79.0%,84.0%,88.0%,12.0%
1,中等,原始基线,42.0%,80.0%,92.0%,92.0%,86.0%,87.6%,52.1%,76.7%,100.0%,84.0%,72.0%,80.0%,20.0%
2,困难,原始基线,18.0%,37.8%,64.4%,66.7%,80.4%,67.1%,37.6%,66.0%,100.0%,74.7%,30.0%,68.0%,35.6%


In [6]:
diagnostic_attribution = pd.DataFrame(report_data["datasets"]["diagnostic_attribution"])
diagnostic_semantic = pd.DataFrame(report_data["datasets"]["diagnostic_semantic"])
display(diagnostic_attribution.style.format({"share": "{:.1%}"}))
display(diagnostic_semantic.style.format({"share": "{:.1%}"}))

,primary,case_count,share
0,intermediate_pipeline_loss,48.000000,52.2%
1,generation_or_reasoning,24.000000,26.1%
2,retrieval_unclear_or_noisy,12.000000,13.0%
3,dataset_reference_issue,3.000000,3.3%
4,evaluator_error,3.000000,3.3%
5,retrieval_insufficient,2.000000,2.2%


,status,case_count,share
0,likely_incorrect_or_incomplete,52.000000,56.5%
1,partially_correct,20.000000,21.7%
2,unresolved,16.000000,17.4%
3,likely_correct,4.000000,4.3%


In [7]:
judge_audit = pd.DataFrame(report_data["datasets"]["semantic_judge"])
display(judge_audit[["item", "result", "status"]])

,item,result,status
0,review coverage,59 unique cases / 59 records,succeeded
1,calibration P3500-EASY-001,expected=False; actual=False,matched
2,calibration P3500-EASY-006,expected=True; actual=True,matched


## Reasonableness Checks

这些检查防止部分运行或未授权 label 被提升为 ready 报告。

In [8]:
baseline_rows = outcomes[outcomes["role"] == "原始基线"]
assert len(baseline_rows) == 3
assert baseline_rows["total"].tolist() == [50, 50, 50]
assert artifact["surface"] == "report"
if FINAL_LABEL is None:
    assert artifact["snapshot"]["status"] == "partial"
    assert not (outcomes["role"] == "最终候选").any()
else:
    assert artifact["snapshot"]["status"] == "ready"
    final_rows = outcomes[outcomes["role"] == "最终候选"]
    assert len(final_rows) == 3 and final_rows["total"].tolist() == [50, 50, 50]
display(pd.DataFrame(report_data["datasets"]["validation_checks"])[["check", "status", "detail"]])

,check,status,detail
0,baseline registry,passed,3 × 50 live root runs
1,baseline artifacts,passed,"copy-manifest, summary and results SHA-256 ver..."
2,diagnostic lineage,passed,diagnostic result/dataset hashes match the reg...
3,semantic judge archive,passed,"summary, selected-case subset and response SHA..."
4,final label selection,pending,no label auto-discovered; provide --final-labe...


## Takeaways

In [9]:
baseline_passes = int(baseline_rows["passed"].sum())
baseline_total = int(baseline_rows["total"].sum())
print(f"Registered baseline: {baseline_passes}/{baseline_total} evaluator-passed cases.")
if FINAL_LABEL is None:
    print("Final comparison is intentionally pending; no remediation-effect claim is supported yet.")
else:
    final_rows = outcomes[outcomes["role"] == "最终候选"]
    final_passes = int(final_rows["passed"].sum())
    final_total = int(final_rows["total"].sum())
    print(f"Explicit final label {FINAL_LABEL}: {final_passes}/{final_total}; delta={final_passes - baseline_passes:+d} cases.")
print("Interpret evaluator pass, deterministic semantic proxy and LLM judge evidence as separate fields.")

Registered baseline: 58/150 evaluator-passed cases.
Final comparison is intentionally pending; no remediation-effect claim is supported yet.
Interpret evaluator pass, deterministic semantic proxy and LLM judge evidence as separate fields.
